# Retail Sales Analytics – Final Load Preparation

## Project Overview
This notebook prepares the cleaned dataset for final export to Tableau Public. We will re-verify data quality, filter to only the essential columns required for our dashboards, generate helper Key Performance Indicator (KPI) fields, and save a highly optimized CSV file ready for visualization.

In [1]:
import pandas as pd
import numpy as np

## 1. Load the Cleaned Dataset

In [2]:
# Loading the processed data
df = pd.read_csv('../data/processed/cleaned_sales.csv')
df['Order Date'] = pd.to_datetime(df['Order Date'])

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Order Year,Order Month,Profit Margin
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,November,16.00
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,November,30.00
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,June,47.00
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,October,-40.00
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,October,11.25


## 2. Recheck Data Quality

Ensuring there are no missing values or incorrect data types before finalizing.

In [3]:
print("Missing Values:\n", df.isnull().sum())
print("\nData Types:\n", df.dtypes)

Missing Values:
 Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
Order Year       0
Order Month      0
Profit Margin    0
dtype: int64

Data Types:
 Row ID                    int64
Order ID                    str
Order Date       datetime64[us]
Ship Date                   str
Ship Mode                   str
Customer ID                 str
Customer Name               str
Segment                     str
Country                     str
City                        str
State                       str
Postal Code               int64
Region                      str
Product ID                  str
Category                    str
Sub-Category        

## 3. Select Required Columns for Dashboard

To optimize dashboard performance, we only retain the columns necessary for visualization.

In [4]:
required_columns = [
    'Order Date', 'Region', 'State', 'Category', 'Sub-Category', 
    'Segment', 'Sales', 'Profit', 'Quantity', 'Discount', 
    'Order Year', 'Order Month', 'Profit Margin'
]

final_df = df[required_columns].copy()
final_df.head()

,Order Date,Region,State,Category,Sub-Category,Segment,Sales,Profit,Quantity,Discount,Order Year,Order Month,Profit Margin
0,2016-11-08,South,Kentucky,Furniture,Bookcases,Consumer,261.9600,41.9136,2,0.00,2016,November,16.00
1,2016-11-08,South,Kentucky,Furniture,Chairs,Consumer,731.9400,219.5820,3,0.00,2016,November,30.00
2,2016-06-12,West,California,Office Supplies,Labels,Corporate,14.6200,6.8714,2,0.00,2016,June,47.00
3,2015-10-11,South,Florida,Furniture,Tables,Consumer,957.5775,-383.0310,5,0.45,2015,October,-40.00
4,2015-10-11,South,Florida,Office Supplies,Storage,Consumer,22.3680,2.5164,2,0.20,2015,October,11.25


## 4. Create KPI Helper Columns

Generating pre-calculated fields for Tableau to simplify dashboard logic:
- **Order Count:** A constant value of 1 to easily aggregate total order line items.
- **Loss Flag:** 1 if Profit < 0, else 0. Highlights unprofitable transactions.
- **High Discount Flag:** 1 if Discount >= 0.3 (30%), else 0. Highlights excessive discounting.

In [5]:
final_df['Order Count'] = 1
final_df['Loss Flag'] = np.where(final_df['Profit'] < 0, 1, 0)
final_df['High Discount Flag'] = np.where(final_df['Discount'] >= 0.3, 1, 0)

final_df[['Profit', 'Loss Flag', 'Discount', 'High Discount Flag']].head()

,Profit,Loss Flag,Discount,High Discount Flag
0,41.9136,0,0.00,0
1,219.5820,0,0.00,0
2,6.8714,0,0.00,0
3,-383.0310,1,0.45,1
4,2.5164,0,0.20,0


## 5. Sort Data Properly

Sorting chronologically by `Order Date` to ensure a logical reading order for BI tools.

In [6]:
final_df = final_df.sort_values(by='Order Date').reset_index(drop=True)
final_df.head()

,Order Date,Region,State,Category,Sub-Category,Segment,Sales,Profit,Quantity,Discount,Order Year,Order Month,Profit Margin,Order Count,Loss Flag,High Discount Flag
0,2014-01-03,Central,Texas,Office Supplies,Paper,Consumer,16.448,5.5512,2,0.2,2014,January,33.75,1,0,0
1,2014-01-04,Central,Illinois,Office Supplies,Labels,Home Office,11.784,4.2717,3,0.2,2014,January,36.25,1,0,0
2,2014-01-04,Central,Illinois,Office Supplies,Storage,Home Office,272.736,-64.7748,3,0.2,2014,January,-23.75,1,1,0
3,2014-01-04,Central,Illinois,Office Supplies,Binders,Home Office,3.540,-5.4870,2,0.8,2014,January,-155.00,1,1,1
4,2014-01-05,East,Pennsylvania,Office Supplies,Art,Consumer,19.536,4.8840,3,0.2,2014,January,25.00,1,0,0


## 6. Save Final Export

Saving the prepared dataset. 

✅ **The `final_dashboard.csv` file is now fully optimized and ready for upload to Tableau Public!**

In [7]:
output_path = '../data/processed/final_dashboard.csv'
final_df.to_csv(output_path, index=False)
print(f"Success! Final dataset saved to {output_path}")

Success! Final dataset saved to ../data/processed/final_dashboard.csv
